In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
!pip install plotly


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df=pd.read_csv("UScomments.csv",on_bad_lines="skip")

In [ ]:
df.head()

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.isnull().sum()

In [ ]:
df=df.dropna()

In [ ]:
df.isnull().sum()

### Sentiment Analysis

In [ ]:
!pip install nltk
import nltk

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
def analyze_comment_sentiment(df):
    compound=sia.polarity_scores(df)["compound"]

    if compound >0.05:
        label = "Positive"
        insight="The comments express Positive emotions"
    elif compound<-0.05:
        label= "Negative"
        insight="The comment express Negative emotions"
    else:
        label = "Neutral"
        insight="The comment express Neutral emotions"
    return {
        "label":label,
        "score": compound,
        "insight": insight }


In [ ]:
df['score']=df['comment_text'].apply(lambda x: analyze_comment_sentiment(x)['score'])

In [ ]:
df.head(5)

In [ ]:
overall_score=df['score'].mean()
print(overall_score)
if overall_score>0.05:
    print("Positive")
elif overall_score<-0.05:
    print("Negative")
else:
    print("Neutral")



### Emoji Analysis


In [ ]:
!pip install emoji==2.15.0

In [ ]:
import emoji

In [ ]:
word="trending 😉"

In [ ]:
word_emoji_list=[]
for char in word:
    if char in emoji.EMOJI_DATA:
        word_emoji_list.append(char)
        

In [ ]:
word_emoji_list

In [ ]:
comment_text_all_emoji=[]
for comment in df['comment_text']:
    for char in comment:
        if char in emoji.EMOJI_DATA:
            comment_text_all_emoji.append(char)

In [ ]:
comment_text_all_emoji

In [ ]:
len(comment_text_all_emoji)

In [ ]:
from collections import Counter

In [ ]:
top_10_emojis=Counter(comment_text_all_emoji).most_common(10)

In [ ]:
emoji_df=pd.DataFrame(top_10_emojis,columns=['Emoji','Count'] )

In [ ]:
import sys
!{sys.executable} -m pip install -U kaleido

In [ ]:
import sys
print(sys.executable)

In [ ]:
!{sys.executable} -m pip show kaleido

In [ ]:
import plotly.express as px

px.bar(emoji_df, x='Emoji', y='Count',
      title='Emoji Analysis Bar Chart',
       height=600
      )



In [ ]:
import plotly
print(plotly.__version__)

### Collect Entire Data Of Youtube

In [ ]:
import os

In [ ]:
file = os.listdir(r"C:\Users\ROHIT\Desktop\Udemy Projects\Youtube Data Analystics\Youtube Analysis\additional_data")

In [ ]:
file

In [ ]:
files_csv=[file for file in file if'.csv' in file]

In [ ]:
files_csv

In [ ]:
full_df=pd.DataFrame()
path=r"C:\Users\ROHIT\Desktop\Udemy Projects\Youtube Data Analystics\Youtube Analysis\additional_data"
for file in files_csv:
    current_df=pd.read_csv(path+'/'+file,encoding='iso-8859-1',on_bad_lines='skip')
    full_df=pd.concat([current_df,full_df],ignore_index=True)
    

In [ ]:
full_df.shape

In [ ]:
full_df[full_df.duplicated()].shape



In [ ]:
full_df=full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
full_df.to_csv(r"C:\Users\ROHIT\Desktop\Udemy Projects\Youtube Data Analystics\Youtube Analysis\Exported Data/Youtube_Whole_data.csv",index=False)

In [ ]:
full_df.to_json(r"C:\Users\ROHIT\Desktop\Udemy Projects\Youtube Data Analystics\Youtube Analysis\Exported Data/Youtube_Whole_data_sample.json",index=False)

### Which Category Dominates Youtube ?


In [ ]:
full_df.dtypes

In [ ]:
full_df['trending_date']=pd.to_datetime(full_df['trending_date'],format='%y.%d.%m')

In [ ]:
full_df.dtypes


In [ ]:
full_df.head()

In [ ]:
import json

In [ ]:
path=r"C:\Users\ROHIT\Desktop\Udemy Projects\Youtube Data Analystics\Youtube Analysis\additional_data\US_category_id.json"

In [ ]:
with open(path,'r',encoding="utf-8") as f:
    data=json.load(f)

In [ ]:
data

In [ ]:
data['items'][0]['snippet']['title']

In [ ]:
data['items'][0]['id']

In [ ]:
cat_dict={}
for item in data['items']:
    cat_dict[int(item['id'])]=item['snippet']['title']

In [ ]:
cat_dict

In [ ]:
full_df['category_name']=full_df['category_id'].map(cat_dict)

In [ ]:
pivot_data=full_df.groupby(['trending_date','category_name'])['views'].sum().unstack()

In [ ]:
pivot_data.head()

In [ ]:
pivot_data.fillna(0)

In [ ]:
area_chart=px.area(pivot_data,
       x=pivot_data.index,y=pivot_data.columns,
       title='Trending Categories Over time',height=600)

In [ ]:
area_chart

In [ ]:
top_5_cat=full_df.groupby('category_name')['views'].sum().nlargest(5).index

In [ ]:
top_5_cat

In [ ]:
top_5=pivot_data[top_5_cat]

In [ ]:
px.area(top_5,
        x=top_5.index,y=top_5.columns,
        title='Top 5 Trending Category Area Chart',height=600)

### Do Viral Video Actually Get Engagement ?


In [ ]:
full_df.head()

In [ ]:
full_df['engagement_rate']=(full_df['likes']+full_df['comment_count'])/full_df['views']

In [ ]:
px.scatter(full_df,
           y='engagement_rate',
           x='views',
          color='category_name',
          hover_name='title',
          size='comment_count',
          size_max=60,
          title='Engagement Bubble Map: Views Vs Engagement Rate',
          height=700,
          log_x=True)

### Views vs Engagement : Inside Youtube's Algorithm !

In [ ]:
category_metrics=full_df.groupby ('category_name').agg(
    total_views=("views","sum"),
    avg_engagement_efficiency=("engagement_rate","mean"),
    video_count=("video_id","count")
).reset_index()

In [ ]:
category_metrics.head()

In [ ]:
px.treemap(category_metrics,path=['category_name'],values='total_views',
           color='avg_engagement_efficiency',
           title='Category Attention Share With Engagement Efficiency Overlay',
               height=600,
          hover_data= {"total_views" : ":,.0f",
                       "avg_engagement_efficiency": ":.3f" ,
                       "video_count":True
          }) 

### Is The Audience Actually Engaged ?

In [ ]:
full_df['engagement_rate'].describe()

In [ ]:
category_engagement_stats=full_df.groupby('category_name')['engagement_rate'].describe()

In [ ]:
px.box(full_df,
       x='category_name',
       y='engagement_rate',
      title='Audience Engagement By Category',
      height=600)

In [ ]:
category_engagement_stats